# Day 014 Solution — RAG with Source Citations

In [ ]:
import re
import ollama
import chromadb
from pydantic import BaseModel, Field

EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL  = "llama3.2"

In [ ]:
# ── helpers ─────────────────────────────────────────────────────────────────

def split_sentences(text: str) -> list[str]:
    """Split text into sentences on . ! ? boundaries."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]


def chunk_by_sentences(
    text: str,
    max_chars: int = 500,
    overlap_sentences: int = 1,
) -> list[str]:
    """Group whole sentences into chunks under max_chars."""
    sentences = split_sentences(text)
    chunks: list[str] = []
    current: list[str] = []
    for sent in sentences:
        candidate = " ".join(current + [sent])
        if current and len(candidate) > max_chars:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] + [sent]
        else:
            current.append(sent)
    if current:
        chunks.append(" ".join(current))
    return chunks


def retrieve_with_sources(
    question: str,
    collection,
    top_k: int = 5,
) -> list[dict]:
    """Retrieve top-k chunks with source metadata and distance scores."""
    q_emb = ollama.embeddings(model=EMBED_MODEL, prompt=question)
    results = collection.query(
        query_embeddings=[q_emb["embedding"]],
        n_results=top_k,
    )
    docs      = results["documents"][0]
    metas     = results["metadatas"][0]
    distances = results["distances"][0]
    return [
        {"text": doc, "source": meta["source"],
         "chunk_index": meta["chunk_index"], "distance": dist}
        for doc, meta, dist in zip(docs, metas, distances)
    ]


def filter_by_relevance(sources: list[dict], max_distance: float = 1.2) -> list[dict]:
    """Keep only chunks whose L2 distance is at or below max_distance."""
    return [s for s in sources if s["distance"] <= max_distance]


CITED_SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer the question using ONLY the numbered "
    "context sources below. Cite sources inline as [1], [2], etc. "
    "If the answer is not in the sources, say: "
    "'I don't know based on the provided documents.'"
)


def build_cited_prompt(question: str, sources: list[dict]) -> str:
    """Build a prompt with numbered, labelled context sources."""
    lines = []
    for i, src in enumerate(sources, start=1):
        lines.append(f"[{i}] ({src['source']})\n{src['text']}")
    context = "\n\n".join(lines)
    return (
        f"Context sources:\n\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer (cite sources as [1], [2], etc.):"
    )


class RagEval(BaseModel):
    faithfulness: float = Field(description="0.0–1.0: every claim supported by context")
    relevance:    float = Field(description="0.0–1.0: answer addresses the question")
    verdict:      str   = Field(description="PASS or FAIL")
    reason:       str   = Field(description="One sentence explaining the verdict")


JUDGE_SYSTEM_PROMPT = """You are an impartial evaluator of RAG answers.
Score from 0.0 to 1.0:
- faithfulness: every claim must be directly supported by the provided context
- relevance: the answer must address the question asked
verdict: PASS if both >= 0.7, else FAIL.
Return JSON matching the schema exactly."""


def evaluate_rag_answer(
    question: str,
    answer: str,
    context_chunks: list[str],
    model: str = CHAT_MODEL,
) -> RagEval:
    """Score a RAG answer for faithfulness and relevance using an LLM judge."""
    schema = RagEval.model_json_schema()
    context = "\n\n---\n\n".join(context_chunks)
    user_msg = (
        f"Schema: {schema}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer: {answer}\n\n"
        f"Evaluate and return JSON."
    )
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        format="json",
    )
    return RagEval.model_validate_json(response["message"]["content"])

In [ ]:
CORPUS = {
    "python.txt": (
        "Python is a high-level, general-purpose programming language created by Guido van Rossum. "
        "It was first released in 1991. Python emphasises code readability and simplicity. "
        "Python supports multiple programming paradigms including procedural, object-oriented, and functional programming. "
        "It uses indentation to delimit blocks rather than curly braces."
    ),
    "chromadb.txt": (
        "ChromaDB is an open-source vector database designed for storing and querying embeddings. "
        "It supports both in-memory and persistent storage modes. "
        "ChromaDB allows metadata filtering alongside vector similarity search. "
        "It is commonly used in RAG pipelines to store document chunks and retrieve them by semantic similarity."
    ),
    "ollama.txt": (
        "Ollama is a tool for running large language models locally on your machine. "
        "It supports models like Llama 3.2, Mistral, and Gemma. "
        "Ollama provides a simple Python API via the ollama package. "
        "The ollama.chat function sends messages to a local model and returns a response dict. "
        "The ollama.embeddings function returns a dense vector representation of a text prompt."
    ),
    "rag.txt": (
        "Retrieval-Augmented Generation (RAG) is a technique that grounds LLM answers in external documents. "
        "The pipeline has three steps: retrieve relevant chunks, augment the prompt with them, then generate. "
        "RAG reduces hallucination by restricting the model to provided context. "
        "Citations in RAG allow users to verify which document each claim came from. "
        "Quality RAG requires good chunking, accurate embeddings, and relevance filtering."
    ),
}

print(f"Corpus: {len(CORPUS)} documents")
for name, text in CORPUS.items():
    print(f"  {name}: {len(text)} chars")

In [ ]:
client = chromadb.Client()
try:
    client.delete_collection("day014_cited_rag")
except Exception:
    pass
collection = client.create_collection("day014_cited_rag")

chunk_count = 0
for source, text in CORPUS.items():
    chunks = chunk_by_sentences(text, max_chars=300, overlap_sentences=1)
    for i, chunk in enumerate(chunks):
        emb = ollama.embeddings(model=EMBED_MODEL, prompt=chunk)["embedding"]
        collection.add(
            ids=[f"{source}_{i:04d}"],
            embeddings=[emb],
            documents=[chunk],
            metadatas=[{"source": source, "chunk_index": i}],
        )
        chunk_count += 1

print(f"Indexed {chunk_count} chunks from {len(CORPUS)} documents")

In [ ]:
def rag_with_citations(
    question: str,
    collection,
    top_k: int = 5,
    max_distance: float = 1.3,
    model: str = CHAT_MODEL,
) -> dict:
    """Full cited RAG: retrieve → filter → prompt → generate → evaluate."""
    sources = retrieve_with_sources(question, collection, top_k=top_k)
    filtered = filter_by_relevance(sources, max_distance=max_distance)
    if not filtered:
        return {
            "answer": "I don't know based on the provided documents.",
            "sources": [],
            "eval": None,
        }
    user_msg = build_cited_prompt(question, filtered)
    messages = [
        {"role": "system", "content": CITED_SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    answer = ollama.chat(model=model, messages=messages)["message"]["content"]
    context_texts = [s["text"] for s in filtered]
    eval_result = evaluate_rag_answer(question, answer, context_texts, model=model)
    return {
        "answer":  answer,
        "sources": filtered,
        "eval":    eval_result,
    }


QUESTIONS = [
    "Who created Python and when was it first released?",
    "What is ChromaDB used for in RAG pipelines?",
    "How does Ollama let you run language models?",
    "What are the three steps of a RAG pipeline?",
    "What is quantum entanglement?",   # out-of-corpus — should return "I don't know"
]

for q in QUESTIONS:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    result = rag_with_citations(q, collection)
    print(f"A: {result['answer']}")
    if result["sources"]:
        print(f"Sources used: {[s['source'] for s in result['sources']]}")
    if result["eval"]:
        ev = result["eval"]
        print(f"Eval: faith={ev.faithfulness:.2f} rel={ev.relevance:.2f} → {ev.verdict}")

In [ ]:
print("Day 014 project complete — RAG with source citations, relevance filtering, and LLM-as-judge evaluation.")